# Chapter 6: Precognition (Thinking Step by Step)

## Lesson

Claude performs significantly better when you give it space to **think before answering**. This is similar to how humans work through problems on paper before giving a final answer.

### Key Insights
- Thinking only helps when it's visible in the output (not hidden)
- You can use XML tags to separate reasoning from the final answer
- This technique is especially powerful for:
  - Classification tasks
  - Logic problems
  - Complex analysis
  - Ambiguous or nuanced questions

### Ordering Bias
Claude can show ordering bias — tending to favor the last option in a list. Step-by-step thinking can counteract this.

In [ ]:
import anthropic

%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt: str = "", prefill: str = ""):
    messages = [{"role": "user", "content": prompt}]
    if prefill:
        messages.append({"role": "assistant", "content": prefill})
    kwargs = {
        "model": MODEL_NAME,
        "max_tokens": 2000,
        "temperature": 0.0,
        "messages": messages
    }
    if system_prompt:
        kwargs["system"] = system_prompt
    message = client.messages.create(**kwargs)
    return message.content[0].text

### Example: Sentiment Analysis with Reasoning

For nuanced text, asking Claude to think through arguments for each side before classifying leads to more accurate results:

In [ ]:
REVIEW = """The new restaurant downtown had amazing ambiance and the appetizers were creative, 
but the main course was overcooked, the service was painfully slow, 
and the bill was much higher than expected for what we got."""

# Without thinking
print("--- Quick classification ---")
response = get_completion(f"Is this review positive or negative?\n\n{REVIEW}")
print(response)
print()

# With step-by-step thinking
print("--- With reasoning ---")
response = get_completion(f"""Classify this review as positive or negative.

<review>{REVIEW}</review>

Before classifying, first write the arguments for positive sentiment in <positive-argument> tags,
then arguments for negative sentiment in <negative-argument> tags.
Finally, give your classification.""")
print(response)

### Example: Using Brainstorm Tags for Factual Recall

In [ ]:
# Without brainstorming
print("--- Direct answer ---")
response = get_completion("Name a famous movie starring an actor who was born in 1956.")
print(response)
print()

# With brainstorming first
print("--- With brainstorming ---")
response = get_completion(
    "Name a famous movie starring an actor who was born in 1956.\n\n"
    "First, brainstorm some actors born in 1956 in <brainstorm> tags. "
    "Then give your final answer."
)
print(response)

---
## Exercises

### Exercise 6.1
Classify the following email into one of these categories:
- **(A)** Pre-sale question
- **(B)** Broken or defective item
- **(C)** Billing question
- **(D)** Other

Output the letter and category name.

In [ ]:
# Exercise 6.1
EMAILS = [
    "Hi — I was charged twice for my order last week. Can you help me get a refund for the duplicate?",
    "I'm interested in your premium plan. Does it include API access?",
    "The widget I received is cracked and doesn't turn on. I need a replacement.",
    "I love your product! Just wanted to say thanks."
]

EXPECTED = ["C", "A", "B", "D"]

for i, email in enumerate(EMAILS):
    PROMPT = f"""Classify this email into one of these categories:
(A) Pre-sale question
(B) Broken or defective item
(C) Billing question
(D) Other

<email>{email}</email>

Output the letter and category name."""
    
    response = get_completion(PROMPT)
    print(f"Email {i+1}: {response[:80]}")
    passed = EXPECTED[i] in response[:5]
    print(f"  Expected: {EXPECTED[i]} | {'✅' if passed else '❌'}\n")

### Exercise 6.2
Same as 6.1, but now wrap **just the letter** in `<answer></answer>` XML tags for easy parsing. Use step-by-step thinking to improve accuracy.

In [ ]:
# Exercise 6.2 - Add thinking and XML answer tags
import re

EMAILS = [
    "Hi — I was charged twice for my order last week. Can you help me get a refund for the duplicate?",
    "I'm interested in your premium plan. Does it include API access?",
    "The widget I received is cracked and doesn't turn on. I need a replacement.",
    "I love your product! Just wanted to say thanks."
]

EXPECTED = ["C", "A", "B", "D"]

for i, email in enumerate(EMAILS):
    # TODO: Modify this prompt to include step-by-step thinking
    # and wrap the final answer letter in <answer> tags
    PROMPT = f"""Classify this email into one of these categories:
(A) Pre-sale question
(B) Broken or defective item
(C) Billing question
(D) Other

<email>{email}</email>

Think through your reasoning step by step, then put your final answer letter in <answer> tags."""
    
    response = get_completion(PROMPT)
    
    # Extract answer from tags
    match = re.search(r"<answer>(.*?)</answer>", response)
    answer = match.group(1).strip() if match else "?"
    
    print(f"Email {i+1}: Answer={answer} | Expected={EXPECTED[i]} | {'✅' if answer == EXPECTED[i] else '❌'}")

---
### Example Playground

In [ ]:
# Playground - try adding thinking steps to any task!
PROMPT = """Is the following statement true or false?

"All mammals lay eggs."

First think through examples in <thinking> tags, then give your answer."""

response = get_completion(PROMPT)
print(response)